# 第7章 债券组合管理 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch07_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch07_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 7：单期免疫演示（U 形）


In [ ]:
import numpy as np
from fi import portfolio as pf, risk, plotting
from fi.cashflow import make_cashflows
plotting.use_chinese_style()
cfs, ts = make_cashflows(0.03, 6, 1, 100); y0 = 0.03
from fi.pricing import price_bond
P0 = price_bond(cfs, ts, y0, 1); H = risk.macaulay_duration(cfs, ts, y0, 1)
target = P0*(1+y0)**H
dys = np.linspace(-0.02, 0.02, 81)
vals = [sum(cf*(1+y0+dy)**(H-t) for cf,t in zip(cfs,ts)) for dy in dys]
print(f'久期=持有期 H={H:.4f}年, 目标终值={target:.4f}')
fig, ax = plotting.new_axes()
ax.plot(dys*100, vals); ax.axhline(target, ls=':', color='gray'); ax.axvline(0, ls=':', color='gray')
ax.set_xlabel('Δy (%)'); ax.set_ylabel(f'H={H:.2f}年 终值'); ax.set_title('单期免疫：Δy=0 处最小')
fig.tight_layout()


## 编程实验 8：现金流匹配 LP + 增删可选债


In [ ]:
liab = [100, 100, 100]
bond_cf = np.array([[102,0,0],[5,105,0],[4,4,104]], dtype=float); prices = [100.0,101.0,100.5]
res = pf.cash_flow_match(liab, bond_cf, prices)
print('3 债：最小成本=', round(res['cost'],4), ' 份数=', np.round(res['units'],4))
# 增加一只更便宜的 3yr 债
bond_cf2 = np.vstack([bond_cf, [3,3,103]]); prices2 = prices + [99.0]
res2 = pf.cash_flow_match(liab, bond_cf2, prices2)
print('4 债：最小成本=', round(res2['cost'],4), ' 份数=', np.round(res2['units'],4))
print('可选券越多→匹配越灵活→成本可能更低')


## 编程实验 9：哑铃久期漂移与再平衡


In [ ]:
ws, wl = pf.two_asset_immunization(1.95, 8.78, 7.0)
print(f'初始哑铃 2Y/10Y 权重 = {ws:.4f}/{wl:.4f}, 组合久期=7')
# 时间+1年&利率+50bp 后，两券久期下降（示意：各减约0.9与0.95）
d_s_new, d_l_new = 1.05, 7.95
d_port = ws*d_s_new + wl*d_l_new
print(f'一年后组合久期漂移到 ≈ {d_port:.3f}（不再=7）')
ws2, wl2 = pf.two_asset_immunization(d_s_new, d_l_new, 7.0)
print(f'恢复久期=7 需再平衡到权重 {ws2:.4f}/{wl2:.4f}——这是免疫的隐性成本')
